# Simple finite-dimensional optimization

Small mathematical programs built as pure `MathematicalProgram` objects and solved with the `Optimizer` method preset `scipy_slsqp`.

Generic **NLP** form used in Minilink:

$$
\begin{aligned}
\min_{z} \quad & J(z) \\
\text{s.t.} \quad & h(z) = 0, \\
& g(z) \geq 0
\end{aligned}
$$

Optional box bounds on $z$ are also supported. Below, $z \in \mathbb{R}^d$ is the decision vector.

In [ ]:
import numpy as np

from minilink.optimization.mathematical_program import MathematicalProgram
from minilink.optimization.optimizer import Optimizer

## Unconstrained quadratic

$$
\min_{z \in \mathbb{R}^{3}} \; \frac{1}{2} \, \lVert z - \bar{z} \rVert_2^2
$$

The minimum is $z^\star = \bar{z}$ (closed form). The analytic gradient is $\nabla J(z) = z - \bar{z}$.

In [ ]:
z_bar = np.array([1.0, -0.5, 2.0])
z0 = np.zeros_like(z_bar)


def J(z: np.ndarray):
    r = z - z_bar
    return 0.5 * (r @ r)


def grad(z: np.ndarray) -> np.ndarray:
    return z - z_bar


unc = MathematicalProgram(n_z=3, J=J, grad_J=grad)
out = Optimizer(
    unc,
    z0=z0,
    method="scipy_slsqp",
    # compile_backend="jax",
    options={"ftol": 1e-12},
).solve(disp=True)

## Convex QP (quadratic objective, linear inequalities)

Standard **QP** form (here with symmetric positive semidefinite $Q$):

$$
\min_{z} \; \frac{1}{2}\, \lVert z \rVert_2^2
\quad \text{s.t.} \quad z_1 \geq 0,\; z_2 \geq 0,\; z_1 + z_2 - 1 \geq 0 .
$$

The unconstrained minimum is $0$ at $z=0$, which violates $z_1+z_2 \geq 1$. The constrained minimum is the point on the line $z_1+z_2=1$ in the first quadrant **closest to the origin**: $z^\star = [\tfrac{1}{2},\,\tfrac{1}{2}]^\top$.

In [ ]:
Q2 = np.eye(2)


def J_qp(z: np.ndarray):
    return 0.5 * (z @ Q2 @ z)


def grad_qp(z: np.ndarray) -> np.ndarray:
    return Q2 @ z


def g_qp(z: np.ndarray) -> np.ndarray:
    return np.array([z[0], z[1], z[0] + z[1] - 1.0])


def jac_qp(z: np.ndarray) -> np.ndarray:
    return np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])


qp = MathematicalProgram(
    n_z=2,
    J=J_qp,
    grad_J=grad_qp,
    g=g_qp,
    jac_g=jac_qp,
)
out = Optimizer(
    qp,
    z0=np.array([0.6, 0.6]),
    method="scipy_slsqp",
    # compile_backend="jax",
    options={"ftol": 1e-12},
).solve(disp=True)